<a href="https://colab.research.google.com/github/Rodkatix/Jewelry-Best-Seller/blob/main/Jewelry_Best_Seller.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **What drives purchase behavior in jewelry**

**Price, material, or trend?**

- A project that predicts or explains which jewelry categories/items sell best and extracts the "why"
- Through feature importance (price point, material, occasion, seasonality, style trend).

**An accurate forecast:**
- Reduction of sales pipeline and forecast risks.
- Reduce the time spent in planning territory coverage and
- Establish benchmarks that can be used to assess future trends.

**Datasets:**
- Using the eCommerce purchase history dataset as the core.
(kaggle/Michael Kechinov)
- Merging data from the Cartier Jewelry Catalog
- Catalog datasets with material/gem-level detail. (kaggle/harshjangid0015)

## **Importing the necessary libraries and overview of the dataset**

In [ ]:
# Libraries to help with reading and manipulating data
import numpy as np
import pandas as pd

# Library to split the data
from sklearn.model_selection import train_test_split

# Libaries to help with data visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Removes the limit for the number of displayed columns
pd.set_option("display.max_columns", None)

# Sets the limit for the number of displayed rows
pd.set_option("display.max_rows", 100)

# Import libraries for building linear regression model
from statsmodels.formula.api import ols
import statsmodels.api as sm
from sklearn.linear_model import LinearRegression

# Import library for preparing data
from sklearn.model_selection import train_test_split

import warnings
warnings.filterwarnings("ignore")

### **Importing the Dataset**

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mkechinov/ecommerce-purchase-history-from-jewelry-store")

print("Path to dataset files:", path)

In [ ]:
import os
jewelry = pd.read_csv(os.path.join(path, "jewelry.csv"))

In [ ]:
# Copying data to another variable to avoid any changes to original data
data = jewelry.copy()

### **View the first and last 5 rows of the dataset**

In [ ]:
data.head()

**Observation:**

- Data is missing descriptive values.
- Let's add an identifier to the columns.

In [ ]:
data.columns = ['order_date_time', 'order_id', 'purchased_product_id', 'quantity_per_order', 'category_id',
'item_description', 'brand_id', 'price_usd', 'user_id', 'item_gender', 'main_color', 'main_metal', 'main_gem']

In [ ]:
data.head(2)

### **Understand the shape of the dataset**

In [ ]:
print(f"There are {data.shape[0]} rows and {data.shape[1]} columns.")

There are 95910 rows and 13 columns.

### **Check the data types of the columns for the dataset**

In [ ]:
data.info()

**Observations:**
- There are 6 object categories including: order date time, and caracteristics of the jewelry as well as item description.
- 4 floats: category brand price and user id.
- 3 int64: product id and quantities per order.

### **Checking for missing values in the dataset**

In [ ]:
# Checking for missing values in the data
data.isnull().sum()

**Observation:**
- There are plenty of missing values.
- Will treat them with the median for numerical and mode for categorical values.

In [ ]:
cols = ['item_description', 'item_gender','main_color', 'main_metal', 'main_gem']

data[cols] = data[cols].fillna(data[cols].mode().iloc[0])


In [ ]:
data.fillna(data.median(numeric_only=True), inplace=True)

In [ ]:
data.isnull().sum()

**Checking for duplicated values**

In [ ]:
# Checking for duplicate values
data.duplicated().sum()

In [ ]:
data[data.duplicated(keep=False)]

In [ ]:
data = data.drop_duplicates()

In [ ]:
data.duplicated().sum()

**Observation:**
- There were 2589 duplicated values.
- We checked the ocurrance of the duplicated rows.
- We dropped the duplicates.

## **Exploratory Data Analysis**

**Let's check the statistical summary of the data.**

In [ ]:
data.describe(include = "all").T

**Observations:**

- The dataset contains 93,321 jewelry transactions.
- Combination of transactional, product, customer, and categorical attributes.
- Price is decidedly right-skewed,
 a mean price of 357.71 usd
compared with a median of 258.77 usd
and a maximum of 34,448.60 usd, indicating the presence of high-value outliers.
- Transaction quantities are almost entirely equal to one, although records with zero quantity should be investigated.
- The categorical variables are highly imbalanced:
 earrings account for approximately 42% of products.
- female items represent approximately 99.6% of records,
- red is the dominant color at approximately 80.6%
- gold represents approximately 98.5% of metals.
- Diamonds are the most common gemstone, accounting for approximately 65.8% of records.

**Let's check the count of each unique category in each of the categorical variables**

In [ ]:
# Making a list of all categorical variables
cat_col = list(data.select_dtypes("object").columns)

# Printing the count of each unique value
for column in cat_col:
    print(data[column].value_counts())
    print("-" * 50)

**Observations:**
- The best seller gems is the diamond with 61385 instances
followed by fianit and topaz
- The most popular metal is gold with 91957 instances
silver is not a close second 1362 and two platinum sells
- The prominant color preferred is jewelry made with the color red, followed by white and yellow.  
- The female market overwhelms with 92962 vs jewelry made for males 359 pieces.
- The best seller is earrings with 38884 instances, followed by 26025 rings
and 13083 pendants.


**Let's separate the information on column "order_data_time" month, date and hour for better readability and potential feature selection.**

In [ ]:
# convert the column from order_data_time to datetime
data['order_date_time'] = pd.to_datetime(data['order_date_time'], utc=True)

In [ ]:
data['date'] = data['order_date_time'].dt.date
data['time'] = data['order_date_time'].dt.time

In [ ]:
data['year'] = data['order_date_time'].dt.year
data['month'] = data['order_date_time'].dt.month
data['day'] = data['order_date_time'].dt.day
data['hour'] = data['order_date_time'].dt.hour
data['day_of_week'] = data['order_date_time'].dt.dayofweek


In [ ]:
data = data.drop(columns=['order_date_time','order_id','user_id'])

In [ ]:
data.info()

In [ ]:
data.tail(7)

**Observation:**
- The order_date_time had too much information in one variable. I separated date and time
- Then featured engineer the date to extract useful pieces of information instead of the raw date
- Drop order_date_time

### **Univariate Analysis**

In [ ]:
# Function to plot a boxplot and a histogram along the same scale

def histogram_boxplot(data, feature, figsize = (12, 7), kde = False, bins = None):
    """
    Boxplot and histogram combined

    data: dataframe
    feature: dataframe column
    figsize: size of figure (default (12,7))
    kde: whether to show density curve (default False)
    bins: number of bins for histogram (default None)
    """
    f2, (ax_box2, ax_hist2) = plt.subplots(
        nrows = 2,      # Number of rows of the subplot grid = 2
        sharex = True,  # x-axis will be shared among all subplots
        gridspec_kw = {"height_ratios": (0.25, 0.75)},
        figsize = figsize,
    )                   # Creating the 2 subplots
    sns.boxplot(
        data = data, x = feature, ax = ax_box2, showmeans = True, color = "violet"
    )                   # Boxplot will be created and a star will indicate the mean value of the column
    sns.histplot(
        data = data, x = feature, kde = kde, ax = ax_hist2, bins = bins, palette = "winter"
    ) if bins else sns.histplot(
        data = data, x = feature, kde = kde, ax = ax_hist2
    )                   # For histogram
    ax_hist2.axvline(
        data[feature].mean(), color = "green", linestyle = "--"
    )                   # Add mean to the histogram
    ax_hist2.axvline(
        data[feature].median(), color = "black", linestyle = "-"
    )                   # Add median to the histogram

- The dataset contains 93,321 jewelry transactions.
- Combination of transactional, product, customer, and categorical attributes.
- Price is decidedly right-skewed,
 a mean price of 357.71 usd
compared with a median of 258.77 usd
and a maximum of 34,448.60 usd, indicating the presence of high-value outliers.
- Transaction quantities are almost entirely equal to one, although records with zero quantity should be investigated.
- The categorical variables are highly imbalanced:
 earrings account for approximately 42% of products.
- female items represent approximately 99.6% of records,
- red is the dominant color at approximately 80.6%
- gold represents approximately 98.5% of metals.
- Diamonds are the most common gemstone, accounting for approximately 65.8% of records.

**Lets check the price**

In [ ]:
histogram_boxplot(data, "price_usd")

**Observation:**
- The price histogram is right-skewed with plenty of outliers whihch might indicate a high-spender sales.

**Saler per year**

In [ ]:
histogram_boxplot(data, "year")

**Observation:**
- Minimal sales on the first two quaters of 2018
- Sales picking up significantly on 2019 surpasing 10,000 usd
- 2019 not recorded sales.
- Sales on 2020 around $25,000 usd

**Sales per month**

In [ ]:
histogram_boxplot(data, "month")

**Observation:**
- Not surprising the most sales on the month of December. Holiday sales.
- Month of August second month in sales. Perhaps heavy discounts.

**Day of the week sales**

In [ ]:
histogram_boxplot(data, "day_of_week")

**Observation:**
- Although not significant variance with the sales per day.
- Day number 3/Wednesday seems to be the most sales per week

**Lets check the nominal data**

In [ ]:
# Function to create labeled barplots

def labeled_barplot(data, feature, perc = False, n = None):
    """
    Barplot with percentage at the top

    data: dataframe
    feature: dataframe column
    perc: whether to display percentages instead of count (default is False)
    n: displays the top n category levels (default is None, i.e., display all levels)
    """

    total = len(data[feature])            # Length of the column
    count = data[feature].nunique()
    if n is None:
        plt.figure(figsize = (count + 1, 5))
    else:
        plt.figure(figsize = (n + 1, 5))

    plt.xticks(rotation = 90, fontsize = 15)
    ax = sns.countplot(
        data = data,
        x = feature,
        palette = "Paired",
        order = data[feature].value_counts().index[:n].sort_values(),
    )

    for p in ax.patches:
        if perc == True:
            label = "{:.1f}%".format(
                100 * p.get_height() / total
            )                              # Percentage of each class of the category
        else:
            label = p.get_height()         # Count of each level of the category

        x = p.get_x() + p.get_width() / 2  # Width of the plot
        y = p.get_height()                 # Height of the plot

        ax.annotate(
            label,
            (x, y),
            ha = "center",
            va = "center",
            size = 12,
            xytext = (0, 5),
            textcoords = "offset points",
        )                                 # Annotate the percentage

    plt.show()                            # Show the plot


**Item Description**

In [ ]:
labeled_barplot(data, "item_description", perc = True)

**The most significant data describing the jewelery pieces is not visible. Lets replace some values with nan.**

In [ ]:
data['item_description'] = data['item_description'].replace(r'^\d+(\.\d+)?$', np.nan, regex=True)

In [ ]:
mode = data['item_description'].mode()[0]

data['item_description'] = data['item_description'].fillna(mode)

In [ ]:
labeled_barplot(data, "item_description", perc = True)

**Observations:**
- Staring from left to right:
- Electronic clocks account for 2% of the sales as well as souveniers same percentage.
- bracelets 6.6 %
- brooches 1.4%
- Earrings out best sellers account for 47.3%
- Followed by rings 27.9, pendants 14% and Necklaces 2.5%
- No studs sold with this dataset.

**Item Gender**

In [ ]:
labeled_barplot(data, "item_gender", perc = True)

**Observations:**
- Female sales account for the overwhelming majority of the purchases with 99.6%.
- Male market accounts only 0.4% which could be an untapped market.

**Predominat Metal**

In [ ]:
labeled_barplot(data, "main_metal", perc = True)

**Observations:**
- Gold surpases sales and preference by 98.5 percent.
- followed by 1.5% in silver.
- and nothing sold in platinum.


**Lets talk about gems**

In [ ]:
labeled_barplot(data, "main_gem", perc = True)

**Observations:**
- Diamonds are the best seller
- finanit (cubic zirconia) 12.8%
- pearls 3.9%
- garnet 2.9 %
- These findings show that customers are willing to spend on diamonds rather than a more affordable pear or garnet.

**Preferred gem color**

In [ ]:
labeled_barplot(data, "main_color", perc = True)

**Observations:**
- Most beloved color is red, followed by white and yellow. This explains garnets having 2.9% sales total

### **Bivariate Analysis**

In [ ]:
cols_list = data.select_dtypes(include = np.number).columns.tolist()

plt.figure(figsize = (10, 5))
sns.heatmap(
    data[cols_list].corr(), annot = True, vmin = -1, vmax = 1, fmt = ".2f", cmap = "Spectral"
)
plt.show()

**Observations:**
The correlation between these variables are notable but not significant.
- Year and brand 0.16 positive correlation
- brand and month 0.11 positive correlation
- brand and purchase product id least correlation with a -0.34 negatice correlation

**Let's check the distribution of our target variable item_description with numeric columns**

In [ ]:
plt.figure(figsize = [8, 6])
sns.scatterplot(y = data.item_description, x = data.year)
plt.show()

**Observation:**
- Product_Weight and Product_Store_Sales_Total are almost linearly correlated with each other.  

In [ ]:
plt.figure(figsize = [10, 8])
sns.scatterplot(x = data.price_usd, y = data.item_description)
plt.show()

**Observation:**
- There seem to be a positive relationship between price and item description.
- Earring and ring are the best seller.
- Ocassional pendant as an high price outlier.

In [ ]:
plt.figure(figsize = [8, 6])
sns.scatterplot(x = data.brand_id, y = data.item_description)
plt.show()

**Observation:**
- There seems no notable correalation betweeen jewelry and brand in general, with the exception of brand and earrings as it seems to play a minor role.

**Lets compare revenue with jewelry items, metals and gems.**

In [ ]:
df_revenue1 = data.groupby(["item_description"], as_index = False)[
    "price_usd"
].sum()
plt.figure(figsize = [14, 8])
plt.xticks(rotation = 90)
a = sns.barplot(x = df_revenue1.item_description, y = df_revenue1.price_usd)
a.set_xlabel("Product Types")
a.set_ylabel("Revenue")
plt.show()

**Observations:
We see consistancy between favorite jewelry and higher revenue**


In [ ]:
df_revenue1 = data.groupby(["main_metal"], as_index = False)[
    "price_usd"
].sum()
plt.figure(figsize = [14, 8])
plt.xticks(rotation = 90)
a = sns.barplot(x = df_revenue1.main_metal, y = df_revenue1.price_usd)
a.set_xlabel("Preferred Metal")
a.set_ylabel("Revenue")
plt.show()

**Observations:**
- Gold is also the favorite metal and the revenue shows sales in the 30,000 usd

**Revenue and gems sales**.

In [ ]:
df_store_revenue = data.groupby(["main_gem"], as_index = False)[
    "price_usd"
].sum()
plt.figure(figsize = [8, 6])
plt.xticks(rotation = 90)
r = sns.barplot(
    x = df_store_revenue.main_gem, y = df_store_revenue.price_usd
)
r.set_xlabel("Preferred Gem")
r.set_ylabel("Revenue")
plt.show()

**Observations:**
- Diamonds remain dominant in sales and revenue compared with the rest of the stones.

**Year and brand**

In [ ]:
df_store_revenue = data.groupby(["year"], as_index = False)[
    "brand_id"
].sum()
plt.figure(figsize = [8, 6])
plt.xticks(rotation = 90)
r = sns.barplot(
    x = df_store_revenue.year, y = df_store_revenue.brand_id
)
r.set_xlabel("year")
r.set_ylabel("Brand")
plt.show()

**Observations:**
- Data set spans from 2018 to 2021
- 2018 show no sales
- 2019 and 2020 show low performing sales
- 2021 exponentially high sales surpasing the 70,000 mark.

**Let's check the distribution of our target variable favorite item with the other categorical columns**

In [ ]:
plt.figure(figsize = [14, 8])
sns.boxplot(y=data.main_gem, x=data.item_description)
plt.xticks(rotation = 90)
plt.title("Price distribution by Gems")
plt.xlabel("Jewelry")
plt.ylabel("Gems")
plt.show()

**Observations**
- Diamonds,Saphire,Amethist are the favorite stones to make jewelry off.
- There are plenty of outliers in all the jewelry categories across all the favorite jewelry pieces for example in pendants and necklaces.
- The pieces of jewelry that customers preferred are earrings and rings with also plenty of outliers favored.

**Jewelry and Date**

In [ ]:
plt.figure(figsize = [14, 8])
sns.boxplot(x=data.item_description, y=data.date)
plt.xticks(rotation = 90)
plt.title("Jewelry and Date")
plt.ylabel("Date")
plt.xlabel("Jewelry")
plt.show()

**Jewelry and Month**

In [ ]:
plt.figure(figsize = [14, 8])
sns.boxplot(y=data.month, x=data.item_description)
plt.xticks(rotation = 90)
plt.title("Favorite jewelry and month purchased")
plt.xlabel("Jewelry")
plt.ylabel("Month")
plt.show()

**Let's find out whether there is some relationship between jewelry and date**
-

In [ ]:
plt.figure(figsize = [14, 8])
sns.boxplot(y=data.main_gem, x=data.price_usd)
plt.xticks(rotation = 90)
plt.title("Price distribution and gems")
plt.ylabel("Type of Gem")
plt.xlabel("Price (USD)")
plt.show()

**Observation:**
- Even when diamonds are the best seller and bring in significant revenue as plenty of outliers are high spenders
- Sitall and rhodolite seem to be popular and bring in revenue
- Mix gems also hold popularity among some customers

**Let's analyze metal and gems in actual numbers**

In [ ]:
plt.figure(figsize = (14, 8))
sns.heatmap(
    pd.crosstab(data["main_gem"], data["main_metal"]),
    annot = True,
    fmt = "g",
    cmap = "viridis",
)
plt.ylabel("main_gem")
plt.xlabel("price_usd")
plt.show()

**Observations**
- Diamonds and gold are the most desired combination totaling 60414 items.
- Fianint(zirconia) and gold total sum of 11740
- Topaz has an edge of 6252

## **Data Preprocessing**

### **Feature Engineering**

**Lets investigate and combine some of these features to answer the following
- When are people buying,
- what are they buying and
- total revenues
- **For that lets import the Cartier dataset to perform keyword level enrichment as a source of additional vocabutary/categories.**

In [ ]:
import kagglehub
path = kagglehub.dataset_download("marcelopesse/cartier-jewelry-catalog")

In [ ]:
import os
cartier = pd.read_csv(os.path.join(path, "cartier_catalog.csv"))


In [ ]:
cartier.head()


In [ ]:
cartier.info()

In [ ]:
metal_keywords = ['yellow gold', 'white gold', 'rose gold', 'platinum', 'silver']

def extract_metal(tag_text, keywords):
    text = str(tag_text).lower()
    for kw in keywords:
        if kw in text:
            return kw
    return 'other/unknown'

cartier['metal'] = cartier['tags'].apply(lambda x: extract_metal(x, metal_keywords))

# Average price by metal type
metal_avg_price = (
    cartier.groupby('metal')['price']
    .agg(['mean', 'count'])
    .sort_values('mean', ascending=False)
)
metal_avg_price



In [ ]:
#cartier = cartier.rename(columns={'description': 'item_description'})

In [ ]:
cartier.info()


**So let us make two broad categories, in order to reduce the number of product types.**

In [ ]:
perishables = [
    "Dairy",
    "Meat",
    "Fruits and Vegetables",
    "Breakfast",
    "Breads",
    "Seafood",
]

In [ ]:
def change(x):
    if x in perishables:
        return "Perishables"
    else:
        return "Non Perishables"


data.Product_Type.apply(change)

In [ ]:
change1 = []
for i in range(0, len(data)):
    if data.Product_Type[i] in perishables:
        change1.append("Perishables")
    else:
        change1.append("Non Perishables")

In [ ]:
data["Product_Type_Category"] = pd.Series(change1)

In [ ]:
data.head()

### **Outlier Check**

- Let's check for outliers in the data.

In [ ]:
# Outlier detection using boxplot
numeric_columns = data.select_dtypes(include=np.number).columns.tolist()
numeric_columns.remove("Store_Establishment_Year")
numeric_columns.remove("Store_Age_Years")


plt.figure(figsize = (15, 12))

for i, variable in enumerate(numeric_columns):
    plt.subplot(4, 4, i + 1)
    plt.boxplot(data[variable], whis = 1.5)
    plt.tight_layout()
    plt.title(variable)

plt.show()

**Observations:**

- There are quite a few outliers in the data.
- However, we will not treat them as they are proper values.

In [ ]:
plt.figure(figsize = (16, 8))
sns.heatmap(data.corr(), annot = True)
plt.show()

**Observation:**
- We observe the high correlation between the two variables gold jewelry and diamonds. Red gems are also preferred

### **Data Preparation for modeling**
- Before we proceed to build a model, we'll have to encode categorical features and drop the unnecessary columns
- We'll split the data into train and test to be able to evaluate the model that we build on the train data.

In [ ]:
data = data.drop(["Product_Type", "Store_Id", "Store_Establishment_Year"], axis = 1)

In [ ]:
data = pd.get_dummies(
    data,
    columns = data.select_dtypes(include = ["object", "category"]).columns.tolist(),
    drop_first = True,
)

In [ ]:
# Separating features and the target column
X = data.drop(["Product_Store_Sales_Total"], axis = 1)
y = data["Product_Store_Sales_Total"]

In [ ]:
X = sm.add_constant(X)

In [ ]:
# Splitting the data into train and test sets in 70:30 ratio
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size = 0.30, random_state = 1)

### **Check for Multicollinearity**

We will use the Variance Inflation Factor (VIF), to check if there is multicollinearity in the data.

Features having a VIF score > 5 will be dropped/treated till all the features have a VIF score < 5

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

# Function to check VIF
def checking_vif(train):
    vif = pd.DataFrame()
    vif["feature"] = train.columns

    # Calculating VIF for each feature
    vif["VIF"] = [
        variance_inflation_factor(train.values, i) for i in range(len(train.columns))
    ]
    return vif

print(checking_vif(X_train))

**Observations:**
- The VIF of Product_Weight, Product_Allocation_Area, Product_MRP, Product_Id_char_FD are less.
- The VIF for dummy variables can be ignored which is expected that they would have a high VIF.
- But the continuous variables should not have high VIF.

In [ ]:
X_train = X_train.drop('Store_Age_Years',axis = 1)

X_test = X_test.drop('Store_Age_Years',axis = 1)

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

# Function to check VIF
def checking_vif(train):
    vif = pd.DataFrame()
    vif["feature"] = train.columns

    # Calculating VIF for each feature
    vif["VIF"] = [
        variance_inflation_factor(train.values, i) for i in range(len(train.columns))
    ]
    return vif


print(checking_vif(X_train))

## **Building Models**

Let's create a function to calculate the performance metrics for our regression model so that we don't need to use the same code repeatedly.

In [ ]:
from sklearn.metrics import r2_score, mean_absolute_percentage_error, mean_absolute_error, mean_squared_error

# Model Performance on test and train data
def model_pref(olsmodel, x_train, x_test):

    # In-sample Prediction
    y_pred_train = olsmodel.predict(x_train)
    y_observed_train = y_train

    # Prediction on test data
    y_pred_test = olsmodel.predict(x_test)
    y_observed_test = y_test

    print(
        pd.DataFrame(
            {
                "Data": ["Train", "Test"],
                "RMSE": [
                    np.sqrt(mean_squared_error(y_pred_train, y_observed_train)),
                    np.sqrt(mean_squared_error(y_pred_test, y_observed_test)),
                ],
                "MAE": [
                    mean_absolute_error(y_pred_train, y_observed_train),
                    mean_absolute_error(y_pred_test, y_observed_test),
                ],

                "r2": [
                    r2_score(y_pred_train, y_observed_train),
                    r2_score(y_pred_test, y_observed_test),
                ],
            }
        )
    )

In [ ]:
# Create the model
model1 = sm.OLS(y_train, X_train).fit()

# Get the model summary
model1.summary()

In [ ]:
# Checking model1 performance
model_pref(model1, X_train, X_test)

**Observations:**
- The Train and the Test scores are very close to each other so we can say the model is not overfitting.
- However, the Test score is slightly better than the Train score. So, we might be able to get better performance if we increase the complexity of the model.

###  **Drop insignificant variables (variables with p-value > 0.05) from the above model and create the regression model again.**

In [ ]:
X_train1 = X_train.drop(["Product_Type_Category_Perishables", "Product_Id_char_FD"], axis = 1)

In [ ]:
X_test1 = X_test.drop(["Product_Type_Category_Perishables", "Product_Id_char_FD"], axis = 1)

In [ ]:
# Create the model
model2 = sm.OLS(y_train, X_train1).fit()

# Get the model summary
model2.summary()

In [ ]:
# Checking model2 performance
model_pref(model2, X_train1, X_test1)

**Observations:**
- The train and the test scores are very close to each other. So, we can say the model is not overfitting.
- However, the test score is slightly better than the training score. So, we might be able to get better performance if we increase the complexity of the model.
- So, model2 is performing the best when compared with model1 because in model2 we are dropping insignificant variables.

### **Checking the below linear regression assumptions**

1. **Mean of residuals should be 0**
2. **No Heteroscedasticity**
3. **Linearity of variables**
4. **Normality of error terms**

### **1. Check for mean residuals**

In [ ]:
residuals = model2.resid

np.mean(residuals)

**Observation:**

- The mean of residuals is very close to 0. Hence, the corresponding assumption is satisfied.

### **2. Check for homoscedasticity**

- Homoscedasticity - If the residuals are symmetrically distributed across the regression line, then the data is said to be homoscedastic.

- Heteroscedasticity- - If the residuals are not symmetrically distributed across the regression line, then the data is said to be heteroscedastic. In this case, the residuals can form a funnel shape or any other non-symmetrical shape.

- We'll use `Goldfeldquandt Test` to test the following hypothesis with alpha = 0.05:

    - Null hypothesis: Residuals are homoscedastic
    - Alternate hypothesis: Residuals have heteroscedastic

In [ ]:
from statsmodels.stats.diagnostic import het_white

from statsmodels.compat import lzip

import statsmodels.stats.api as sms

In [ ]:
import statsmodels.stats.api as sms

from statsmodels.compat import lzip

name = ["F statistic", "p-value"]

test = sms.het_goldfeldquandt(y_train, X_train1)

lzip(name, test)

**Observation:**

- Since p-value > 0.05, we cannot reject the Null Hypothesis that the residuals are homoscedastic and the corresponding assumption is satisfied.

### **3. Linearity of variables**

It states that the predictor variables must have a linear relation with the dependent variable.

To test the assumption, we'll plot residuals and the fitted values on a plot and ensure that residuals do not form a strong pattern. They should be randomly and uniformly scattered on the x-axis.

In [ ]:
# Predicted values
fitted = model2.fittedvalues

# sns.set_style("whitegrid")
sns.residplot(x = fitted, y = residuals, color = "lightblue", lowess = True)

plt.xlabel("Fitted Values")

plt.ylabel("Residual")

plt.title("Residual PLOT")

plt.show()

**Observation:**

- There is no pattern in the residual vs fitted values plot. Hence, the corresponding assumption is satisfied.

### **4. Normality of error terms**

The residuals should be normally distributed.

In [ ]:
# Plot histogram of residuals
sns.histplot(residuals, kde = True)

In [ ]:
# Plot q-q plot of residuals
import pylab

import scipy.stats as stats

stats.probplot(residuals, dist = "norm", plot = pylab)

plt.show()

**Observation:**

- From the above plots, the residuals seem to follow a normal distribution. Hence, the corresponding assumption is satisfied. Now, we will check the model performance on the train and test datasets.

### **Apply cross validation to improve the model and evaluate it using different evaluation metrics**

Let's check the performance of the model using the cross-validation technique from the scikit-learn library and see if the performance on the train and the test data is comparable to what we are getting after cross-validating the data.

In [ ]:
# Import the required function

from sklearn.model_selection import cross_val_score

# Build the regression model and cross-validate
linearregression = LinearRegression()

cv_Score11 = cross_val_score(linearregression, X_train, y_train, cv = 10)
cv_Score12 = cross_val_score(linearregression, X_train, y_train, cv = 10,
                             scoring = 'neg_mean_squared_error')


print("RSquared: %0.3f (+/- %0.3f)" % (cv_Score11.mean(), cv_Score11.std() * 2))
print("Mean Squared Error: %0.3f (+/- %0.3f)" % (-1*cv_Score12.mean(), cv_Score12.std() * 2))

**Observation:**
- After applying cross-validation the model score has improved. We can compare it by the evaluation metric scores.

## **Actionable Insights and Business Recommendations**

**To recap the original question of price material or trend. Lets explore the findings**

- The dataset contains 93,321 jewelry transactions.
- Combination of transactional, product, customer, and categorical attributes.
- Plenty of high spending outliers spending a total of  20% of total revenue. These are high valued customers who should be encouraged and persuaded with extra perks.
- One purchased with transaction perhaps. Room for discount of second item.
- Earrings account for approximately 42% of products.
Female items sold represent approximately 99.6% of records
- Red is the dominant color at approximately 80.6%
gold represents approximately 98.5% of metals.
Diamonds are the most common gemstone, accounting for approximately 65.8% of records.
- The best seller gems is the diamond with 61385 instances, followed by fianit(zirconia) and topaz
- The prominent color preferred is jewelry made with the color red, followed by white and yellow.  
- Not surprising the most sales on the month of December. Holiday sales.Month of August second month in sales. Perhaps heavy discounts.
- Day number 3/Wednesday seems to be the most sales per week
Male market accounts only 0.4% which could be an untapped market?.
These findings show that customers are willing to spend on diamonds rather than a more affordable pear or garnet.
- Even when diamonds are the best seller and bring in significant revenue as plenty of outliers are high spenders.
- Sitall and rhodolite seem to be popular and bring in revenue.
- Mix gems also hold popularity among some customers.


**Additional information that can be collected to gain better insights -**

- Customers' details like age and gender can be incorporated in this model so that the company gets to know their target audience well and can build their sales strategies according to that.

- The company should also keep a watch for the number of festive occasions present in a quarter so that they can strategize their inventory accordingly.